In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.pipeline import Pipeline

In [2]:
data = pd.read_csv('wrapper_dataset_5000.csv')

In [3]:
print(data.head())

   age  income  experience  score  click_rate  time_spent    noise1    noise2  \
0   58   83865          23     70        0.13          45  0.146598  0.282067   
1   48   76686           6     59        0.20          52  0.656071  0.284284   
2   34   34675          21     94        0.28          36  0.558102  0.018300   
3   27   96706          16     58        0.13          46  0.842640  0.692431   
4   40   40068           5     87        0.26          61  0.183253  0.775948   

   target  
0       0  
1       1  
2       1  
3       0  
4       1  


In [4]:
print(data.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 9 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   age         5000 non-null   int64  
 1   income      5000 non-null   int64  
 2   experience  5000 non-null   int64  
 3   score       5000 non-null   int64  
 4   click_rate  5000 non-null   float64
 5   time_spent  5000 non-null   int64  
 6   noise1      5000 non-null   float64
 7   noise2      5000 non-null   float64
 8   target      5000 non-null   int64  
dtypes: float64(3), int64(6)
memory usage: 351.7 KB
None


In [5]:
print(data.describe())

               age        income   experience        score   click_rate  \
count  5000.000000   5000.000000  5000.000000  5000.000000  5000.000000   
mean     39.726200  64490.403600    12.709800    71.542600     0.177002   
std      11.506299  20318.387846     6.866834    13.020111     0.071930   
min      20.000000  30002.000000     1.000000    50.000000     0.050000   
25%      30.000000  46708.000000     7.000000    60.000000     0.120000   
50%      40.000000  64268.500000    13.000000    71.000000     0.180000   
75%      50.000000  81661.750000    19.000000    83.000000     0.240000   
max      59.000000  99996.000000    24.000000    94.000000     0.300000   

        time_spent       noise1       noise2       target  
count  5000.000000  5000.000000  5000.000000  5000.000000  
mean     49.457400     0.510139     0.497847     0.578400  
std      17.147379     0.289210     0.287153     0.493865  
min      20.000000     0.000021     0.000102     0.000000  
25%      35.000000     0

In [6]:

X = data.drop(columns=['target'])
y = data['target']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(max_iter=1000))
])

pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)

baseline_acc = accuracy_score(y_test, y_pred)

In [7]:
forward_selector = SequentialFeatureSelector(
    pipeline,
    n_features_to_select='auto',
    direction='forward',
    scoring='accuracy',
    cv=5
)

forward_selector.fit(X_train, y_train)

forward_features = X.columns[forward_selector.get_support()]

pipeline.fit(X_train[forward_features], y_train)
y_pred = pipeline.predict(X_test[forward_features])
forward_acc = accuracy_score(y_test, y_pred)


In [8]:
backward_selector = SequentialFeatureSelector(
    pipeline,
    n_features_to_select='auto',
    direction='backward',
    scoring='accuracy',
    cv=5
)

backward_selector.fit(X_train, y_train)

backward_features = X.columns[backward_selector.get_support()]

pipeline.fit(X_train[backward_features], y_train)
y_pred = pipeline.predict(X_test[backward_features])
backward_acc = accuracy_score(y_test, y_pred)


In [9]:
print("Baseline Accuracy:", baseline_acc)

print("\nForward Selected Features:", list(forward_features))
print("Forward Selection Accuracy:", forward_acc)

print("\nBackward Selected Features:", list(backward_features))
print("Backward Elimination Accuracy:", backward_acc)

print("\n PERFORMANCE COMPARISON =")
print("Baseline Accuracy        :", baseline_acc)
print("Forward Selection Acc    :", forward_acc)
print("Backward Elimination Acc :", backward_acc)


Baseline Accuracy: 0.876

Forward Selected Features: ['income', 'score', 'click_rate', 'time_spent']
Forward Selection Accuracy: 0.878

Backward Selected Features: ['income', 'score', 'click_rate', 'time_spent']
Backward Elimination Accuracy: 0.878

 PERFORMANCE COMPARISON =
Baseline Accuracy        : 0.876
Forward Selection Acc    : 0.878
Backward Elimination Acc : 0.878
